# Notebook 08: Transfer Learning with Pre-trained Audio Models

**Purpose:** Apply transfer learning using state-of-the-art pre-trained audio models

**Objectives:**
1. Load pre-trained models (YAMNet, VGGish, OpenL3)
2. Extract embeddings from angle grinder audio
3. Fine-tune top layers for binary classification
4. Compare with custom models from notebook 07
5. Evaluate deployment feasibility

**Pre-trained Models:**
- **YAMNet**: Google's audio event classifier (AudioSet-521 classes)
- **VGGish**: Audio embedding model from Google
- **OpenL3**: Open-source audio/image embeddings

**Transfer Learning Benefits:**
- Leverage models trained on millions of audio samples
- Better generalization with limited data
- Often superior to training from scratch
- Can achieve high accuracy with less training time

**Expected Outcome:**
- Higher accuracy than custom models
- Better feature representations
- Potentially larger model size (trade-off)

---

## Section 1: Setup & Configuration

In [1]:
import os, sys
from pathlib import Path
import json
import time
from datetime import datetime
import warnings
import gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Audio processing
import librosa
import soundfile as sf

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras import backend as K

# TensorFlow Hub for pre-trained models
import tensorflow_hub as hub

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print(f'TensorFlow version: {tf.__version__}')
print(f'TensorFlow Hub version: {hub.__version__}')
print(f'Librosa version: {librosa.__version__}')
print('✓ All libraries imported')

TensorFlow version: 2.17.1
TensorFlow Hub version: 0.16.1
Librosa version: 0.10.2.post1
✓ All libraries imported


In [5]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features' / 'neural'
MODELS_DIR = PROJECT_ROOT / 'models' / 'transfer_learning'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root:  {PROJECT_ROOT}')
print(f'Data dir:      {DATA_DIR}')
print(f'Models dir:    {MODELS_DIR}')
print(f'Results dir:   {RESULTS_DIR}')

Project root:  /Users/harryirving/Development/projects/ai-ml/BikeAIv5
Data dir:      /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed
Models dir:    /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/transfer_learning
Results dir:   /Users/harryirving/Development/projects/ai-ml/BikeAIv5/results


In [6]:
# Set random seeds
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# TensorFlow GPU configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'✓ GPU memory growth enabled for {len(gpus)} GPU(s)')
    except RuntimeError as e:
        print(f'GPU configuration error: {e}')

print(f'✓ Random seed set to {SEED}')

✓ Random seed set to 42


## Section 2: Load Audio Data

In [7]:
print('\nLOADING AUDIO DATA...')
print('='*70)

# Load audio file paths and labels
grinder_dir = DATA_DIR / 'grinder'
non_grinder_dir = DATA_DIR / 'non_grinder'

# Collect file paths
audio_files = []
labels = []

# Grinder samples
if grinder_dir.exists():
    grinder_files = list(grinder_dir.glob('*.wav'))
    audio_files.extend(grinder_files)
    labels.extend([1] * len(grinder_files))
    print(f'✓ Found {len(grinder_files)} grinder samples')

# Non-grinder samples
if non_grinder_dir.exists():
    non_grinder_files = list(non_grinder_dir.glob('*.wav'))
    audio_files.extend(non_grinder_files)
    labels.extend([0] * len(non_grinder_files))
    print(f'✓ Found {len(non_grinder_files)} non-grinder samples')

if len(audio_files) == 0:
    raise FileNotFoundError(f'No audio files found in {DATA_DIR}')

# Convert to arrays
audio_files = np.array(audio_files)
labels = np.array(labels)

print(f'\nTotal samples: {len(audio_files):,}')
print(f'  Grinder:     {(labels == 1).sum():,}')
print(f'  Non-grinder: {(labels == 0).sum():,}')
print(f'  Balance:     {(labels == 1).sum() / len(labels) * 100:.1f}% grinder')

print('='*70)


LOADING AUDIO DATA...


FileNotFoundError: No audio files found in /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed

In [ ]:
print('\nSPLITTING DATA...')
print('='*70)

# Split: 70% train, 15% val, 15% test
files_temp, files_test, y_temp, y_test = train_test_split(
    audio_files, labels, test_size=0.15, random_state=SEED, stratify=labels
)

files_train, files_val, y_train, y_val = train_test_split(
    files_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
)

print(f'Train: {len(files_train):,} files')
print(f'Val:   {len(files_val):,} files')
print(f'Test:  {len(files_test):,} files')

# Calculate class weights
n_grinder = (y_train == 1).sum()
n_non_grinder = (y_train == 0).sum()
total = len(y_train)

class_weight = {
    0: total / (2 * n_non_grinder),
    1: total / (2 * n_grinder)
}

print(f'\nClass weights:')
print(f'  Non-grinder: {class_weight[0]:.3f}')
print(f'  Grinder:     {class_weight[1]:.3f}')

print('='*70)

## Section 3: YAMNet - Google's Audio Event Classifier

### 3.1 Load YAMNet Model

In [ ]:
print('\nLOADING YAMNET MODEL...')
print('='*70)

# YAMNet from TensorFlow Hub
YAMNET_MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'

print(f'Loading YAMNet from TensorFlow Hub...')
print(f'URL: {YAMNET_MODEL_HANDLE}')

yamnet_model = hub.load(YAMNET_MODEL_HANDLE)

print('✓ YAMNet loaded successfully')
print('\nYAMNet specifications:')
print('  - Input: 16kHz mono audio (any duration)')
print('  - Output: 1024-dim embeddings per 0.96s frame')
print('  - Pre-trained on AudioSet (521 classes)')
print('  - Architecture: MobileNet-v1')

print('='*70)

### 3.2 Extract YAMNet Embeddings

In [ ]:
def extract_yamnet_embeddings(audio_file, target_sr=16000):
    """
    Extract YAMNet embeddings from audio file.
    
    Args:
        audio_file: Path to audio file
        target_sr: Target sample rate (YAMNet expects 16kHz)
    
    Returns:
        embeddings: Mean of frame embeddings (1024-dim)
    """
    # Load audio
    audio, sr = librosa.load(audio_file, sr=target_sr, mono=True)
    
    # Ensure proper shape for YAMNet
    audio = audio.astype(np.float32)
    
    # Get embeddings from YAMNet
    scores, embeddings, spectrogram = yamnet_model(audio)
    
    # Average embeddings across frames
    # YAMNet outputs embeddings for each 0.96s frame
    # We take the mean to get a single vector per audio file
    embedding_mean = tf.reduce_mean(embeddings, axis=0).numpy()
    
    return embedding_mean

print('✓ YAMNet embedding function defined')

In [ ]:
print('\nEXTRACTING YAMNET EMBEDDINGS...')
print('='*70)

# Check if embeddings already exist
yamnet_embeddings_file = FEATURES_DIR / 'yamnet_embeddings.npy'
yamnet_labels_file = FEATURES_DIR / 'yamnet_labels.npy'

if yamnet_embeddings_file.exists() and yamnet_labels_file.exists():
    print('Loading cached YAMNet embeddings...')
    X_yamnet = np.load(yamnet_embeddings_file)
    y_all = np.load(yamnet_labels_file)
    print(f'✓ Loaded cached embeddings: {X_yamnet.shape}')
else:
    print('Extracting embeddings from audio files...')
    print('This will take several minutes...')
    
    embeddings_list = []
    
    for i, audio_file in enumerate(audio_files):
        if i % 100 == 0:
            print(f'  Processed {i}/{len(audio_files)} files...')
        
        try:
            embedding = extract_yamnet_embeddings(audio_file)
            embeddings_list.append(embedding)
        except Exception as e:
            print(f'Error processing {audio_file}: {e}')
            # Use zero embedding as fallback
            embeddings_list.append(np.zeros(1024))
    
    X_yamnet = np.array(embeddings_list)
    y_all = labels
    
    # Save embeddings
    np.save(yamnet_embeddings_file, X_yamnet)
    np.save(yamnet_labels_file, y_all)
    
    print(f'\n✓ Embeddings extracted and saved')

print(f'\nYAMNet embeddings shape: {X_yamnet.shape}')
print(f'Labels shape: {y_all.shape}')
print(f'Memory usage: {X_yamnet.nbytes / 1024**2:.1f} MB')

print('='*70)

### 3.3 Split YAMNet Data

In [ ]:
print('\nSPLITTING YAMNET EMBEDDINGS...')
print('='*70)

# Use same split indices as before
X_yamnet_temp, X_yamnet_test, y_yamnet_temp, y_yamnet_test = train_test_split(
    X_yamnet, y_all, test_size=0.15, random_state=SEED, stratify=y_all
)

X_yamnet_train, X_yamnet_val, y_yamnet_train, y_yamnet_val = train_test_split(
    X_yamnet_temp, y_yamnet_temp, test_size=0.176, random_state=SEED, stratify=y_yamnet_temp
)

print(f'Train: {X_yamnet_train.shape[0]:,} samples')
print(f'Val:   {X_yamnet_val.shape[0]:,} samples')
print(f'Test:  {X_yamnet_test.shape[0]:,} samples')

# Normalize embeddings
scaler_yamnet = StandardScaler()
X_yamnet_train_scaled = scaler_yamnet.fit_transform(X_yamnet_train)
X_yamnet_val_scaled = scaler_yamnet.transform(X_yamnet_val)
X_yamnet_test_scaled = scaler_yamnet.transform(X_yamnet_test)

print('\n✓ Embeddings normalized')

# Memory cleanup
del X_yamnet, X_yamnet_temp
gc.collect()

print('='*70)

### 3.4 Build Classifier on YAMNet Embeddings

In [ ]:
def build_yamnet_classifier(input_dim=1024, dropout_rate=0.4):
    """
    Build a classifier on top of YAMNet embeddings.
    
    Simple MLP head for binary classification.
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout_rate),
        
        layers.Dense(1, activation='sigmoid')
    ], name='YAMNet_Classifier')
    
    return model

print('✓ YAMNet classifier architecture defined')

### 3.5 Train YAMNet Classifier

In [ ]:
# Custom F1 metric
class F1Score(keras.metrics.Metric):
    def __init__(self, name='f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision = keras.metrics.Precision()
        self.recall = keras.metrics.Recall()
    
    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)
    
    def result(self):
        p = self.precision.result()
        r = self.recall.result()
        return 2 * ((p * r) / (p + r + K.epsilon()))
    
    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()

print('✓ F1 metric defined')

In [ ]:
print('\nTRAINING YAMNET CLASSIFIER...')
print('='*70)

# Build model
yamnet_classifier = build_yamnet_classifier()

# Compile
yamnet_classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        F1Score(name='f1')
    ]
)

yamnet_classifier.summary()

# Callbacks
checkpoint = callbacks.ModelCheckpoint(
    str(MODELS_DIR / 'yamnet_classifier_best.keras'),
    monitor='val_f1',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stop = callbacks.EarlyStopping(
    monitor='val_f1',
    mode='max',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

print('\nStarting training...')
start_time = time.time()

# Train
history_yamnet = yamnet_classifier.fit(
    X_yamnet_train_scaled, y_yamnet_train,
    batch_size=32,
    epochs=50,
    validation_data=(X_yamnet_val_scaled, y_yamnet_val),
    class_weight=class_weight,
    callbacks=[checkpoint, early_stop, reduce_lr],
    verbose=1
)

train_time = time.time() - start_time

print(f'\n✓ Training complete in {train_time/60:.1f} minutes')
print(f'  Best val F1: {max(history_yamnet.history["val_f1"]):.4f}')

print('='*70)

### 3.6 Evaluate YAMNet Classifier

In [ ]:
print('\nEVALUATING YAMNET CLASSIFIER...')
print('='*70)

# Predictions
start_time = time.time()
y_pred_yamnet = yamnet_classifier.predict(X_yamnet_test_scaled, verbose=0)
yamnet_inference_time = (time.time() - start_time) / len(X_yamnet_test_scaled) * 1000

y_pred_yamnet_binary = (y_pred_yamnet > 0.5).astype(int).flatten()

# Metrics
yamnet_acc = accuracy_score(y_yamnet_test, y_pred_yamnet_binary)
yamnet_f1 = f1_score(y_yamnet_test, y_pred_yamnet_binary)
yamnet_precision = precision_score(y_yamnet_test, y_pred_yamnet_binary)
yamnet_recall = recall_score(y_yamnet_test, y_pred_yamnet_binary)
yamnet_auc = roc_auc_score(y_yamnet_test, y_pred_yamnet)

print('YAMNet Transfer Learning Results:')
print(f'  Accuracy:  {yamnet_acc:.4f}')
print(f'  F1 Score:  {yamnet_f1:.4f}')
print(f'  Precision: {yamnet_precision:.4f}')
print(f'  Recall:    {yamnet_recall:.4f}')
print(f'  AUC:       {yamnet_auc:.4f}')
print(f'  FNR:       {(1 - yamnet_recall) * 100:.2f}%')
print(f'  Inference: {yamnet_inference_time:.2f} ms/sample')

# Confusion matrix
cm_yamnet = confusion_matrix(y_yamnet_test, y_pred_yamnet_binary)
print(f'\nConfusion Matrix:')
print(cm_yamnet)

print('='*70)

## Section 4: Fine-tuning Strategy (Optional Advanced Approach)

In [ ]:
print('\nFINE-TUNING YAMNET (OPTIONAL)...')
print('='*70)
print('\nNote: Fine-tuning YAMNet end-to-end requires:')
print('  - More computational resources')
print('  - Longer training time')
print('  - Risk of overfitting with small datasets')
print('\nFor this project, feature extraction + classifier is recommended.')
print('\nTo implement fine-tuning:')
print('  1. Create a KerasLayer from YAMNet with trainable=True')
print('  2. Add classification head')
print('  3. Freeze YAMNet initially')
print('  4. Train head for several epochs')
print('  5. Unfreeze top layers of YAMNet')
print('  6. Continue training with lower learning rate')
print('\nSkipping for now - classifier approach is sufficient.')
print('='*70)

## Section 5: Compare with Custom Models from Notebook 07

In [ ]:
print('\nCOMPARING WITH CUSTOM MODELS...')
print('='*70)

# Load results from notebook 07
nb07_report_3ch = RESULTS_DIR / '07_neural_recommendation_report_3ch.json'
nb07_report_1ch = RESULTS_DIR / '07_neural_recommendation_report_1ch.json'

comparison_data = []

# Add YAMNet results
comparison_data.append({
    'Model': 'YAMNet Transfer Learning',
    'Feature Type': 'YAMNet Embeddings (1024-dim)',
    'Accuracy': yamnet_acc,
    'F1': yamnet_f1,
    'Precision': yamnet_precision,
    'Recall': yamnet_recall,
    'AUC': yamnet_auc,
    'FNR (%)': (1 - yamnet_recall) * 100,
    'Inference (ms)': yamnet_inference_time,
    'Params': yamnet_classifier.count_params()
})

# Load custom model results
if nb07_report_3ch.exists():
    with open(nb07_report_3ch, 'r') as f:
        nb07_3ch = json.load(f)
    
    best = nb07_3ch['best_neural_model']
    comparison_data.append({
        'Model': f"{best['model_name']} (Custom)",
        'Feature Type': '3-channel Spectrograms',
        'Accuracy': best['test_accuracy'],
        'F1': best['test_f1'],
        'Precision': best['precision'],
        'Recall': best['recall'],
        'AUC': best['auc'],
        'FNR (%)': best['fnr_percent'],
        'Inference (ms)': best['inference_ms'],
        'Params': best['parameters']
    })

if nb07_report_1ch.exists():
    with open(nb07_report_1ch, 'r') as f:
        nb07_1ch = json.load(f)
    
    best = nb07_1ch['best_neural_model']
    comparison_data.append({
        'Model': f"{best['model_name']} (Custom)",
        'Feature Type': '1-channel Spectrograms',
        'Accuracy': best['test_accuracy'],
        'F1': best['test_f1'],
        'Precision': best['precision'],
        'Recall': best['recall'],
        'AUC': best['auc'],
        'FNR (%)': best['fnr_percent'],
        'Inference (ms)': best['inference_ms'],
        'Params': best['parameters']
    })

# Create comparison DataFrame
df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.sort_values('F1', ascending=False)

print('\nCOMPARISON OF ALL MODELS:')
print('='*70)
print(df_comparison.to_string(index=False))
print('='*70)

# Save comparison
comparison_csv = RESULTS_DIR / '08_transfer_learning_comparison.csv'
df_comparison.to_csv(comparison_csv, index=False)
print(f'\n✓ Saved comparison: {comparison_csv.name}')

# Determine best model
best_model = df_comparison.iloc[0]
print(f'\n🏆 BEST MODEL: {best_model["Model"]}')
print(f'  F1 Score: {best_model["F1"]:.4f}')
print(f'  Recall:   {best_model["Recall"]:.4f}')

print('='*70)

## Section 6: Visualization

In [ ]:
print('\nGenerating visualizations...')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Training history
axes[0, 0].plot(history_yamnet.history['loss'], label='Train Loss', alpha=0.7)
axes[0, 0].plot(history_yamnet.history['val_loss'], label='Val Loss', linestyle='--', alpha=0.7)
axes[0, 0].set_title('YAMNet Training Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(history_yamnet.history['f1'], label='Train F1', alpha=0.7)
axes[0, 1].plot(history_yamnet.history['val_f1'], label='Val F1', linestyle='--', alpha=0.7)
axes[0, 1].set_title('YAMNet F1 Score', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 2. Confusion matrix
sns.heatmap(cm_yamnet, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0])
axes[1, 0].set_title('YAMNet Confusion Matrix', fontweight='bold')
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_xticklabels(['Non-grinder', 'Grinder'])
axes[1, 0].set_yticklabels(['Non-grinder', 'Grinder'])

# 3. Model comparison
if len(df_comparison) > 1:
    models_short = [m.split('(')[0].strip()[:15] for m in df_comparison['Model']]
    x = np.arange(len(models_short))
    width = 0.25
    
    axes[1, 1].bar(x - width, df_comparison['F1'], width, label='F1', alpha=0.8)
    axes[1, 1].bar(x, df_comparison['Recall'], width, label='Recall', alpha=0.8)
    axes[1, 1].bar(x + width, df_comparison['Precision'], width, label='Precision', alpha=0.8)
    
    axes[1, 1].set_xlabel('Model')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].set_title('Model Comparison', fontweight='bold')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(models_short, rotation=45, ha='right')
    axes[1, 1].legend()
    axes[1, 1].grid(axis='y', alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'Run Notebook 07\nfor comparison',
                   ha='center', va='center', fontsize=14)
    axes[1, 1].set_title('Model Comparison', fontweight='bold')

plt.tight_layout()
fig_path = FIGURES_DIR / '08_transfer_learning_results.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {fig_path.name}')
plt.close()

print('✓ Visualization complete')

## Section 7: Save Results & Recommendations

In [ ]:
print('\nSAVING FINAL REPORT...')
print('='*70)

# Create report
report = {
    'timestamp': datetime.now().isoformat(),
    'notebook': '08_transfer_learning_yamnet',
    'yamnet_results': {
        'model': 'YAMNet Transfer Learning',
        'embeddings_dim': 1024,
        'classifier_params': int(yamnet_classifier.count_params()),
        'test_accuracy': float(yamnet_acc),
        'test_f1': float(yamnet_f1),
        'precision': float(yamnet_precision),
        'recall': float(yamnet_recall),
        'auc': float(yamnet_auc),
        'fnr_percent': float((1 - yamnet_recall) * 100),
        'inference_ms': float(yamnet_inference_time),
        'training_time_minutes': float(train_time / 60)
    },
    'best_overall_model': {
        'name': best_model['Model'],
        'feature_type': best_model['Feature Type'],
        'f1': float(best_model['F1']),
        'recall': float(best_model['Recall'])
    },
    'recommendations': []
}

# Add recommendations
if best_model['Model'].startswith('YAMNet'):
    report['recommendations'].append('YAMNet transfer learning outperforms custom models')
    report['recommendations'].append('Consider YAMNet for deployment if size permits')
    report['recommendations'].append('YAMNet requires full audio processing pipeline')
else:
    report['recommendations'].append('Custom models perform better or similarly to YAMNet')
    report['recommendations'].append('Custom models are more efficient for ESP32-S3')
    report['recommendations'].append('Proceed with custom model quantization')

report['recommendations'].append('Next: Quantize best model for ESP32-S3 (Notebook 09)')

# Save report
report_path = RESULTS_DIR / '08_transfer_learning_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'✓ Saved report: {report_path.name}')

# Save YAMNet classifier
yamnet_classifier.save(MODELS_DIR / 'yamnet_classifier_final.keras')
print(f'✓ Saved YAMNet classifier')

# Save scaler
import joblib
joblib.dump(scaler_yamnet, MODELS_DIR / 'scaler_yamnet.pkl')
print(f'✓ Saved scaler')

print('\n' + '='*70)
print('✓ NOTEBOOK 08 COMPLETE')
print('='*70)

print(f'\nYAMNet Transfer Learning Results:')
print(f'  F1 Score:  {yamnet_f1:.4f}')
print(f'  Recall:    {yamnet_recall:.4f}')
print(f'  Inference: {yamnet_inference_time:.2f} ms')

print(f'🏆 Best Overall Model: {best_model["Model"]}')
print(f'  F1 Score:  {best_model["F1"]:.4f}')
print(f'  Recall:    {best_model["Recall"]:.4f}')

print('\nNext Steps:')
print('  1. Review comparison results')
print('  2. Choose deployment model (YAMNet vs Custom)')
print('  3. Proceed to Notebook 09: Model Quantization')
print('  4. Deploy to ESP32-S3 (Notebook 10)')

print('='*70)

# Memory cleanup
del X_yamnet_train_scaled, X_yamnet_val_scaled, X_yamnet_test_scaled
gc.collect()
K.clear_session()
print('✓ Memory cleaned up')